# Modeling Pipeline — LightGBM Two-Stage Strategy

**Objectif** : Selectionner les opportunites FTR profitables pour le mois cible M+1.

**Strategie** :
1. **Classificateur** LightGBM (`scale_pos_weight`) → P(profitable) par opportunite
2. **Regresseur** LightGBM (`log1p(PROFIT)`) → magnitude du profit predit
3. **Selection** : filtre par seuil de proba → rang par profit predit → top-K, K ∈ [10, 100]

**Donnees** : `data/master_dataset.parquet` (211K lignes, 3,173 EIDs, 71 colonnes)

**Validation** : Walk-forward temporel par trimestre (pas de k-fold — interdit pour donnees temporelles)

In [ ]:
# Cell 0 — Setup & Installation
# Installer les packages manquants si necessaire
import subprocess, importlib

for pkg, import_name in [('lightgbm', 'lightgbm'), ('optuna', 'optuna'), ('shap', 'shap')]:
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call(['pip', 'install', pkg, '-q'])

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import optuna
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
np.random.seed(SEED)

DATA_PATH = 'data/master_dataset.parquet'
print("Setup complete.")

In [ ]:
# Cell 1 — Load Data & Define Feature Columns
df = pd.read_parquet(DATA_PATH)
print(f"Dataset complet: {len(df):,} lignes, {df.shape[1]} colonnes")
print(f"Taux positif global: {df['TARGET'].mean()*100:.2f}%")

# Exclure sim-only du training (0% positif — pas de PR/C)
df_model = df[df['is_sim_only'] == 0].copy()
print(f"\nPool market-validated: {len(df_model):,} lignes")
print(f"Taux positif market: {df_model['TARGET'].mean()*100:.2f}%")

# Colonnes par role
ID_COLS = ['EID', 'MONTH', 'PEAKID', 'DECISION_MONTH']
TARGET_COLS = ['TARGET', 'PROFIT', 'PR', 'PR_signed', 'C']
META_COLS = ['is_sim_only', 'season']  # season_encoded utilise a la place de season (str)

FEATURE_COLS = [c for c in df_model.columns
                if c not in ID_COLS + TARGET_COLS + META_COLS]

print(f"\nFeatures ({len(FEATURE_COLS)}): {FEATURE_COLS}")

In [ ]:
# Cell 2 — Temporal Train / Validation / Test Split
# Walk-forward temporel :
#   Train:      2020-01 → 2022-06 (30 mois)
#   Validation: 2022-07 → 2023-06 (12 mois, walk-forward par trimestre)
#   Test:       2023-07 → 2023-12 (6 mois, intouchable jusqu'a l'eval finale)

train_mask = df_model['MONTH'] < '2022-07'
val_mask   = (df_model['MONTH'] >= '2022-07') & (df_model['MONTH'] < '2023-07')
test_mask  = df_model['MONTH'] >= '2023-07'

X_train = df_model.loc[train_mask, FEATURE_COLS]
y_train = df_model.loc[train_mask, ['TARGET', 'PROFIT']]
X_val   = df_model.loc[val_mask, FEATURE_COLS]
y_val   = df_model.loc[val_mask, ['TARGET', 'PROFIT']]
X_test  = df_model.loc[test_mask, FEATURE_COLS]
y_test  = df_model.loc[test_mask, ['TARGET', 'PROFIT']]

# Metadata pour evaluation (PR, C pour calculer le profit net reel)
val_meta  = df_model.loc[val_mask, ID_COLS + ['PR', 'C']].copy()
test_meta = df_model.loc[test_mask, ID_COLS + ['PR', 'C']].copy()

print("=== Split temporel ===")
for name, mask in [('Train', train_mask), ('Val', val_mask), ('Test', test_mask)]:
    n = mask.sum()
    pos = int(df_model.loc[mask, 'TARGET'].sum())
    months = df_model.loc[mask, 'MONTH']
    print(f"{name:5s}: {n:>7,} lignes | {pos:>5,} positifs ({pos/n*100:.1f}%) | {months.min()} → {months.max()}")

In [ ]:
# Cell 3 — Fonctions d'evaluation
def evaluate_selection(y_true_target, y_true_profit, probas, pred_profit,
                       months, threshold=0.5, K=50):
    """
    Evalue une strategie de selection mois par mois.
    
    Pour chaque mois :
    1. Filtre les candidats dont proba >= threshold
    2. Parmi eux, selectionne les top-K par profit predit
    3. Enforce la contrainte 10-100 opportunites
    
    Returns: (F1 agrege, profit net total, details par mois)
    """
    results = []
    all_selected = []
    all_actuals = []
    
    for month in sorted(months.unique()):
        m_idx = months == month
        m_proba = probas[m_idx.values]
        m_pred_profit = pred_profit[m_idx.values]
        m_target = y_true_target[m_idx].values
        m_profit = y_true_profit[m_idx].values
        
        # Etape 1 : filtrer par seuil de proba
        candidates = np.where(m_proba >= threshold)[0]
        if len(candidates) == 0:
            # Aucun candidat : prendre les top-K par proba
            candidates = np.argsort(m_proba)[-min(K, len(m_proba)):]
        
        # Etape 2 : trier par profit predit, garder top-K
        if len(candidates) > K:
            top_k_idx = candidates[np.argsort(m_pred_profit[candidates])[-K:]]
        else:
            top_k_idx = candidates
        
        # Etape 3 : contrainte 10-100
        if len(top_k_idx) > 100:
            top_k_idx = top_k_idx[np.argsort(m_pred_profit[top_k_idx])[-100:]]
        elif len(top_k_idx) < 10:
            remaining = np.setdiff1d(np.arange(len(m_proba)), top_k_idx)
            n_extra = min(10 - len(top_k_idx), len(remaining))
            if n_extra > 0:
                extra = remaining[np.argsort(m_proba[remaining])[-n_extra:]]
                top_k_idx = np.concatenate([top_k_idx, extra])
        
        selected = np.zeros(len(m_target), dtype=int)
        selected[top_k_idx] = 1
        
        all_selected.extend(selected)
        all_actuals.extend(m_target)
        
        # Profit net = somme des PROFIT reels des opportunites selectionnees
        net_profit = m_profit[top_k_idx].sum()
        n_tp = int((m_target[top_k_idx] == 1).sum())
        results.append({
            'month': month, 'n_selected': len(top_k_idx),
            'n_tp': n_tp, 'net_profit': net_profit
        })
    
    f1 = f1_score(all_actuals, all_selected, zero_division=0)
    total_profit = sum(r['net_profit'] for r in results)
    return f1, total_profit, results


def print_results(f1, profit, results, label=""):
    """Affiche un resume des resultats."""
    print(f"\n{'='*50}")
    print(f"  {label}")
    print(f"  F1-score: {f1:.4f}  |  Profit net: {profit:>12,.0f}")
    print(f"{'='*50}")
    for r in results:
        print(f"  {r['month']}: {r['n_selected']:>3} selectionnes, "
              f"{r['n_tp']:>2} TP, profit={r['net_profit']:>10,.0f}")

print("Fonctions d'evaluation definies.")

## Baseline — Regression Logistique

Premier modele simple pour etablir un plancher de performance. Si LightGBM ne bat pas la regression logistique, c'est un red flag.

In [ ]:
# Cell 4 — Baseline : Regression Logistique
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED)
lr.fit(X_train_scaled, y_train['TARGET'])

lr_proba = lr.predict_proba(X_val_scaled)[:, 1]
# Proxy de profit naif : proba * pr_lag1 (meilleur signal disponible)
lr_pred_profit = lr_proba * X_val['pr_lag1'].values

val_months = df_model.loc[val_mask, 'MONTH']

# Chercher le meilleur seuil pour la baseline
best_lr_f1 = 0
best_lr_t = 0.5
for t in np.arange(0.1, 0.8, 0.05):
    f1, profit, _ = evaluate_selection(
        y_val['TARGET'], y_val['PROFIT'], lr_proba, lr_pred_profit,
        val_months, threshold=t, K=50)
    if f1 > best_lr_f1:
        best_lr_f1 = f1
        best_lr_t = t

lr_f1, lr_profit, lr_results = evaluate_selection(
    y_val['TARGET'], y_val['PROFIT'], lr_proba, lr_pred_profit,
    val_months, threshold=best_lr_t, K=50)

print_results(lr_f1, lr_profit, lr_results,
              f"BASELINE — Regression Logistique (seuil={best_lr_t:.2f})")

## LightGBM — Classificateur + Regresseur

Deux modeles avec parametres par defaut raisonnables, avant optimisation Optuna.

In [ ]:
# Cell 5 — LightGBM Classificateur
scale_pos = (y_train['TARGET'] == 0).sum() / (y_train['TARGET'] == 1).sum()
print(f"scale_pos_weight = {scale_pos:.2f}")

clf_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'verbosity': -1,
    'seed': SEED,
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'max_depth': 6,
    'num_leaves': 31,
    'min_child_samples': 50,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 1.0,
    'reg_lambda': 1.0,
    'scale_pos_weight': scale_pos,
}

clf = lgb.LGBMClassifier(**clf_params)
clf.fit(
    X_train, y_train['TARGET'],
    eval_set=[(X_val, y_val['TARGET'])],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

clf_proba = clf.predict_proba(X_val)[:, 1]
print(f"\nClassifieur entraine — {clf.best_iteration_} iterations")

In [ ]:
# Cell 6 — LightGBM Regresseur (cible = log1p(PROFIT))
# PROFIT est tres asymetrique (median=342, max=212K) → log1p compresse les extremes
y_train_profit_log = np.log1p(np.maximum(y_train['PROFIT'], 0))
y_val_profit_log   = np.log1p(np.maximum(y_val['PROFIT'], 0))

reg_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'verbosity': -1,
    'seed': SEED,
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'max_depth': 6,
    'num_leaves': 31,
    'min_child_samples': 50,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 1.0,
    'reg_lambda': 1.0,
}

reg = lgb.LGBMRegressor(**reg_params)
reg.fit(
    X_train, y_train_profit_log,
    eval_set=[(X_val, y_val_profit_log)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

reg_pred_log = reg.predict(X_val)
reg_pred_profit = np.expm1(reg_pred_log)  # retour a l'echelle originale
print(f"\nRegresseur entraine — {reg.best_iteration_} iterations")

## Comparaison des 3 approches

- **A** : Regresseur seul → seuil sur profit predit > 0 → top-K
- **B** : Classificateur seul → proba > seuil → top-K par proba
- **C** : Two-stage (classificateur filtre + regresseur rank)

In [ ]:
# Cell 7 — Comparaison des 3 approches sur validation
val_months = df_model.loc[val_mask, 'MONTH']

# Approche A : Regresseur seul
# Le profit predit sert a la fois de filtre (>0) et de rang
reg_proba_proxy = np.clip(reg_pred_profit / (reg_pred_profit.max() + 1e-8), 0, 1)
a_f1, a_profit, a_res = evaluate_selection(
    y_val['TARGET'], y_val['PROFIT'],
    reg_proba_proxy, reg_pred_profit, val_months, threshold=0.01, K=50)

# Approche B : Classificateur seul (rang par proba)
b_f1, b_profit, b_res = evaluate_selection(
    y_val['TARGET'], y_val['PROFIT'],
    clf_proba, clf_proba, val_months, threshold=0.3, K=50)

# Approche C : Two-stage (classificateur filtre + regresseur rank)
c_f1, c_profit, c_res = evaluate_selection(
    y_val['TARGET'], y_val['PROFIT'],
    clf_proba, reg_pred_profit, val_months, threshold=0.3, K=50)

print("=" * 65)
print(f"{'Approche':<30} {'F1':>8} {'Profit net':>15}")
print("-" * 65)
print(f"{'A — Regresseur seul':<30} {a_f1:>8.4f} {a_profit:>15,.0f}")
print(f"{'B — Classificateur seul':<30} {b_f1:>8.4f} {b_profit:>15,.0f}")
print(f"{'C — Two-stage (clf+reg)':<30} {c_f1:>8.4f} {c_profit:>15,.0f}")
print(f"{'Baseline — LogReg':<30} {lr_f1:>8.4f} {lr_profit:>15,.0f}")
print("=" * 65)

## Optimisation du seuil et de K

Grid search sur la validation pour trouver le couple (threshold, K) optimal.

In [ ]:
# Cell 8 — Optimisation threshold + K sur validation
# Utilise l'approche C (two-stage) comme base
best_score = -np.inf
best_params = {}
grid_results = []

for threshold in np.arange(0.05, 0.70, 0.05):
    for K in [10, 15, 20, 30, 40, 50, 75, 100]:
        f1, profit, _ = evaluate_selection(
            y_val['TARGET'], y_val['PROFIT'],
            clf_proba, reg_pred_profit, val_months,
            threshold=threshold, K=K)
        grid_results.append({
            'threshold': round(threshold, 2), 'K': K,
            'f1': f1, 'profit': profit
        })
        if f1 > best_score:
            best_score = f1
            best_params = {'threshold': round(threshold, 2), 'K': K,
                          'f1': f1, 'profit': profit}

print(f"Meilleur couple: threshold={best_params['threshold']:.2f}, K={best_params['K']}")
print(f"F1={best_params['f1']:.4f}, Profit={best_params['profit']:,.0f}")

# Heatmap F1 par (threshold, K)
grid_df = pd.DataFrame(grid_results)
pivot = grid_df.pivot_table(values='f1', index='threshold', columns='K')
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[0])
axes[0].set_title('F1-score par (threshold, K)')

pivot_profit = grid_df.pivot_table(values='profit', index='threshold', columns='K')
sns.heatmap(pivot_profit, annot=True, fmt=',.0f', cmap='RdYlGn', ax=axes[1])
axes[1].set_title('Profit net par (threshold, K)')

plt.tight_layout()
plt.show()

## Optuna — Optimisation des hyperparametres

Walk-forward CV trimestriel (4 folds) pour eviter toute fuite temporelle.
Objectif : maximiser le F1-score moyen sur les 4 folds de validation.

In [ ]:
# Cell 9 — Optuna : optimisation des hyperparametres du classificateur
# Walk-forward CV : 4 folds trimestriels (expanding window)

FOLDS = [
    ('2022-07', '2022-10'),  # Q3 2022
    ('2022-10', '2023-01'),  # Q4 2022
    ('2023-01', '2023-04'),  # Q1 2023
    ('2023-04', '2023-07'),  # Q2 2023
]

def objective(trial):
    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'verbosity': -1,
        'seed': SEED,
        'n_estimators': 1500,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'num_leaves': trial.suggest_int('num_leaves', 15, 63),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 150),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'scale_pos_weight': scale_pos,
    }
    
    cv_scores = []
    for val_start, val_end in FOLDS:
        tr = df_model[(df_model['MONTH'] < val_start)]
        vl = df_model[(df_model['MONTH'] >= val_start) & (df_model['MONTH'] < val_end)]
        
        if len(vl) == 0:
            continue
        
        model = lgb.LGBMClassifier(**params)
        model.fit(
            tr[FEATURE_COLS], tr['TARGET'],
            eval_set=[(vl[FEATURE_COLS], vl['TARGET'])],
            callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)]
        )
        
        proba = model.predict_proba(vl[FEATURE_COLS])[:, 1]
        
        # Meilleur F1 avec seuil optimal pour ce fold
        best_f1 = 0
        for t in np.arange(0.1, 0.6, 0.05):
            preds = (proba >= t).astype(int)
            f1 = f1_score(vl['TARGET'], preds, zero_division=0)
            best_f1 = max(best_f1, f1)
        cv_scores.append(best_f1)
    
    return np.mean(cv_scores) if cv_scores else 0.0

print("Lancement Optuna (100 trials, walk-forward CV)...")
study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=100, show_progress_bar=True)

print(f"\nMeilleur F1 CV: {study.best_value:.4f}")
print(f"Meilleurs parametres:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

## Modeles finaux & Evaluation sur le test set

Entrainement des modeles finaux avec les meilleurs hyperparametres sur train+val, evaluation sur test (2023-H2).

In [ ]:
# Cell 10 — Entrainement des modeles finaux avec meilleurs params Optuna
# Classificateur final : best Optuna params, entraine sur train + val
best_clf_params = {
    **study.best_params,
    'objective': 'binary',
    'metric': 'binary_logloss',
    'verbosity': -1,
    'seed': SEED,
    'n_estimators': 1500,
    'scale_pos_weight': scale_pos,
}

trainval_mask = ~test_mask
X_trainval = df_model.loc[trainval_mask, FEATURE_COLS]
y_trainval_target = df_model.loc[trainval_mask, 'TARGET']
y_trainval_profit = np.log1p(np.maximum(df_model.loc[trainval_mask, 'PROFIT'], 0))

final_clf = lgb.LGBMClassifier(**best_clf_params)
final_clf.fit(X_trainval, y_trainval_target)
print(f"Classificateur final entraine sur {len(X_trainval):,} lignes")

# Regresseur final : memes parametres de base (peut etre optimise separement)
final_reg_params = {
    **{k: v for k, v in study.best_params.items()
       if k not in ['scale_pos_weight']},
    'objective': 'regression',
    'metric': 'rmse',
    'verbosity': -1,
    'seed': SEED,
    'n_estimators': 1500,
}

final_reg = lgb.LGBMRegressor(**final_reg_params)
final_reg.fit(X_trainval, y_trainval_profit)
print(f"Regresseur final entraine sur {len(X_trainval):,} lignes")

In [ ]:
# Cell 11 — Evaluation finale sur le test set (2023-H2)
test_proba = final_clf.predict_proba(X_test)[:, 1]
test_pred_profit = np.expm1(final_reg.predict(X_test))

test_f1, test_profit, test_monthly = evaluate_selection(
    y_test['TARGET'], y_test['PROFIT'],
    test_proba, test_pred_profit,
    df_model.loc[test_mask, 'MONTH'],
    threshold=best_params['threshold'], K=best_params['K'])

print_results(test_f1, test_profit, test_monthly,
              f"TEST FINAL (2023-H2) — threshold={best_params['threshold']:.2f}, K={best_params['K']}")

# Classification report standard
test_preds = (test_proba >= best_params['threshold']).astype(int)
print(f"\n--- Classification Report (seuil={best_params['threshold']:.2f}) ---")
print(classification_report(y_test['TARGET'], test_preds, target_names=['Non-profitable', 'Profitable']))

# Matrice de confusion
cm = confusion_matrix(y_test['TARGET'], test_preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Non-profitable', 'Profitable'],
            yticklabels=['Non-profitable', 'Profitable'])
ax.set_xlabel('Predit')
ax.set_ylabel('Reel')
ax.set_title(f'Matrice de confusion — Test (seuil={best_params["threshold"]:.2f})')
plt.tight_layout()
plt.show()

## Feature Importance & Interpretabilite

Analyse des features les plus importantes pour le jury (50% de la note).

In [ ]:
# Cell 12 — Feature Importance
importance_clf = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': final_clf.feature_importances_
}).sort_values('importance', ascending=False)

importance_reg = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': final_reg.feature_importances_
}).sort_values('importance', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 10))

# Classificateur
importance_clf.head(20).plot.barh(
    x='feature', y='importance', ax=axes[0], color='steelblue', legend=False)
axes[0].set_title('Top 20 Features — Classificateur')
axes[0].invert_yaxis()

# Regresseur
importance_reg.head(20).plot.barh(
    x='feature', y='importance', ax=axes[1], color='coral', legend=False)
axes[1].set_title('Top 20 Features — Regresseur')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print("\n=== Top 10 Classificateur ===")
for _, row in importance_clf.head(10).iterrows():
    print(f"  {row['feature']:30s} {row['importance']:>6.0f}")

print("\n=== Top 10 Regresseur ===")
for _, row in importance_reg.head(10).iterrows():
    print(f"  {row['feature']:30s} {row['importance']:>6.0f}")

In [ ]:
# Cell 13 — SHAP values (si disponible)
try:
    import shap
    
    # Echantillon pour SHAP (limite pour la performance)
    shap_sample = X_test.sample(min(1000, len(X_test)), random_state=SEED)
    
    explainer = shap.TreeExplainer(final_clf)
    shap_values = explainer.shap_values(shap_sample)
    
    # Pour les modeles binaires, shap_values peut etre une liste [neg, pos]
    sv = shap_values[1] if isinstance(shap_values, list) else shap_values
    
    plt.figure(figsize=(12, 8))
    shap.summary_plot(sv, shap_sample, max_display=20, show=False)
    plt.title('SHAP Summary — Classificateur (impact sur P(profitable))')
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("SHAP non installe — pip install shap pour activer cette visualisation")
except Exception as e:
    print(f"Erreur SHAP: {e}")

## Export des predictions

Generation du fichier `opportunities.csv` au format attendu par le jury.

In [ ]:
# Cell 14 — Export des predictions au format soumission
# Appliquer le modele final sur le test set avec les meilleurs parametres

output = test_meta.copy()
output['predicted_proba'] = test_proba
output['predicted_profit'] = test_pred_profit
output['selected'] = 0

for month in sorted(output['MONTH'].unique()):
    mask = output['MONTH'] == month
    m_data = output[mask]
    
    # Filtrer par seuil de proba
    candidates = m_data[m_data['predicted_proba'] >= best_params['threshold']]
    
    # Si pas assez de candidats, prendre les top-10 par proba
    if len(candidates) < 10:
        candidates = m_data.nlargest(10, 'predicted_proba')
    
    # Selectionner top-K par profit predit
    K = min(best_params['K'], len(candidates))
    K = max(10, K)  # minimum 10
    selected = candidates.nlargest(K, 'predicted_profit')
    
    output.loc[selected.index, 'selected'] = 1

# Verifier la contrainte 10-100 par mois
print("=== Verification contrainte 10-100 par mois ===")
for month in sorted(output['MONTH'].unique()):
    n = output[(output['MONTH'] == month) & (output['selected'] == 1)].shape[0]
    status = "OK" if 10 <= n <= 100 else "VIOLATION"
    print(f"  {month}: {n:>3} selectionnes [{status}]")

# Formater pour soumission
submission = output[output['selected'] == 1][['MONTH', 'PEAKID', 'EID']].copy()
submission.columns = ['TARGET_MONTH', 'PEAK_TYPE', 'EID']
submission['PEAK_TYPE'] = submission['PEAK_TYPE'].map({0: 'OFF', 1: 'ON'})

submission.to_csv('data/opportunities.csv', index=False)
print(f"\nExporte {len(submission)} opportunites dans data/opportunities.csv")
print(submission.head(10))

## Resume des resultats

Tableau comparatif de toutes les approches testees.

In [ ]:
# Cell 15 — Resume comparatif
print("=" * 70)
print(f"{'RESUME DES RESULTATS':^70}")
print("=" * 70)
print(f"\n{'Modele':<35} {'Split':>8} {'F1':>8} {'Profit':>12}")
print("-" * 70)
print(f"{'Baseline — LogReg':<35} {'Val':>8} {lr_f1:>8.4f} {lr_profit:>12,.0f}")
print(f"{'A — Regresseur seul':<35} {'Val':>8} {a_f1:>8.4f} {a_profit:>12,.0f}")
print(f"{'B — Classificateur seul':<35} {'Val':>8} {b_f1:>8.4f} {b_profit:>12,.0f}")
print(f"{'C — Two-stage (clf+reg)':<35} {'Val':>8} {c_f1:>8.4f} {c_profit:>12,.0f}")
print(f"{'C optimise (t={best_params[\"threshold\"]:.2f}, K={best_params[\"K\"]})':<35} {'Val':>8} {best_params['f1']:>8.4f} {best_params['profit']:>12,.0f}")
print(f"{'Final Optuna + optimise':<35} {'Test':>8} {test_f1:>8.4f} {test_profit:>12,.0f}")
print("=" * 70)

print(f"\n--- Configuration finale ---")
print(f"  Seuil de proba:  {best_params['threshold']:.2f}")
print(f"  K par mois:      {best_params['K']}")
print(f"  scale_pos_weight: {scale_pos:.2f}")
print(f"  Meilleurs params Optuna: {study.best_params}")
print(f"\n  Opportunites exportees: {len(submission)}")
print(f"  Fichier: data/opportunities.csv")